<a href="https://colab.research.google.com/github/DanieldeSSilva/Curso-python/blob/main/Assistente_Virtual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assistente Virtual

Instalar dependências no Colab

In [18]:
# Instalar dependências no Colab
!pip install pyttsx3
!pip install SpeechRecognition
!pip install geopy
!apt-get install -y espeak-ng
!apt-get install -y portaudio19-dev
!pip install pyaudio


import pyttsx3
import speech_recognition as sr
import webbrowser
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests


# Inicializar o mecanismo de texto para fala
engine = pyttsx3.init()
def speak(text):
    engine.say(text)
    engine.runAndWait()

# Função para capturar áudio do microfone
def get_audio():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        speak("Estou ouvindo...")
        try:
            audio = recognizer.listen(source)
            command = recognizer.recognize_google(audio, language="pt-BR")
            print(f"Você disse: {command}")
            return command.lower()
        except sr.UnknownValueError:
            speak("Desculpe, não entendi o que você disse.")
        except sr.RequestError as e:
            speak(f"Erro no serviço de reconhecimento de fala: {e}")
    return ""

# Funções automatizadas
def buscar_wikipedia():
    speak("O que você gostaria de pesquisar na Wikipedia?")
    query = get_audio()
    if query:
        url = f"https://pt.wikipedia.org/wiki/{query.replace(' ', '_')}"
        webbrowser.open(url)
        speak(f"Abrindo a pesquisa na Wikipedia para {query}")

def abrir_youtube():
    webbrowser.open("https://www.youtube.com")
    speak("Abrindo o YouTube")

def encontrar_farmacia():
    speak("Buscando a farmácia mais próxima.")
    geolocator = Nominatim(user_agent="assistente_virtual_colab")
    try:
        location = geolocator.geocode("São Paulo, Brasil")  # Simulação de localização fixa
        if location:
            latitude, longitude = location.latitude, location.longitude
            url = f"https://nominatim.openstreetmap.org/search?q=farmacia&format=json&addressdetails=1&limit=5&lat={latitude}&lon={longitude}&radius=5000"
            response = requests.get(url)
            if response.status_code == 200:
                farmacias = response.json()
                if farmacias:
                    farmacia_mais_proxima = farmacias[0]
                    nome = farmacia_mais_proxima['display_name']
                    coordenadas = (farmacia_mais_proxima['lat'], farmacia_mais_proxima['lon'])
                    distancia = geodesic((latitude, longitude), coordenadas).km
                    speak(f"A farmácia mais próxima é {nome}, a cerca de {distancia:.2f} quilômetros de você.")
                    webbrowser.open(f"https://www.google.com/maps/dir/{latitude},{longitude}/{farmacia_mais_proxima['lat']},{farmacia_mais_proxima['lon']}")
                else:
                    speak("Não foi possível encontrar farmácias próximas.")
            else:
                speak("Houve um erro ao tentar localizar farmácias próximas.")
        else:
            speak("Não foi possível determinar sua localização atual.")
    except Exception as e:
        speak(f"Erro ao tentar encontrar farmácias: {e}")

# Função principal do assistente
def assistente_virtual():
    speak("Olá, eu sou o seu assistente virtual. Como posso ajudar?")
    while True:
        command = get_audio()
        if "wikipedia" in command:
            buscar_wikipedia()
        elif "youtube" in command:
            abrir_youtube()
        elif "farmácia" in command:
            encontrar_farmacia()
        elif "sair" in command:
            speak("Até logo!")
            break
        else:
            speak("Desculpe, não reconheci o comando. Tente novamente.")

if __name__ == "__main__":
    assistente_virtual()
